In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


In [3]:
def load_pareto_history(filepath="pareto_history.pkl"):
    """
    Carga el archivo pickle que contiene fronts_history.
    Devuelve un dict: {generacion: {nivel_frente: [registros...]}}
    """
    with open(filepath, "rb") as f:
        history = pickle.load(f)
    return history

In [22]:
def pareto_history_to_df(filepath="pareto_history.pkl", generation=0):
    """
    Converts Pareto fronts history for a given generation into a DataFrame
    with columns: generation, front_level, accuracy, params, inference_time.
    """
    history = load_pareto_history(filepath)
    if generation not in history:
        raise ValueError(f"Generation {generation} not found in history.")
    
    records = []
    for level, recs in history[generation].items():
        for rec in recs:
            records.append({
                "generation": generation,
                "front_level": level,
                "accuracy": rec["accuracy"],
                "params": rec["params"],
                "inference_time": rec["inference_time"]
            })
    df = pd.DataFrame(records)
    return df

In [28]:
def plot_pareto_evolution(history, dims="3d",
                        x="params", y="inference_time", z="accuracy",
                        width=1200, height=800, y_range=None):
    """
    Plot the evolution of Pareto fronts over generations in 2D or 3D using Plotly.
    
    Args:
        history (dict): Pareto history as loaded by load_pareto_history.
        dims (str): '2d' or '3d' for plot dimensionality.
        x (str): Column name for x-axis.
        y (str): Column name for y-axis.
        z (str): Column name for z-axis (ignored if dims='2d').
        width (int): Figure width in pixels.
        height (int): Figure height in pixels.
        y_range (list or tuple): Optional fixed range for the y-axis [min, max].
    """
    # Flatten history into a DataFrame
    rows = []
    for gen, fronts in history.items():
        for level, recs in fronts.items():
            for rec in recs:
                rows.append({
                    "generation": gen,
                    "front_level": level,
                    "accuracy": rec["accuracy"],
                    "params": rec["params"],
                    "inference_time": rec["inference_time"]
                })
    df = pd.DataFrame(rows)
    
    if dims == "3d":
        fig = px.scatter_3d(
            df, x=x, y=y, z=z,
            color="front_level",
            animation_frame="generation",
            width=width, height=height,
            title="Pareto Front Evolution (3D)",
            labels={
                x: x.replace('_',' ').title(),
                y: y.replace('_',' ').title(),
                z: z.replace('_',' ').title(),
                "front_level": "Front Level",
                "generation": "Generation"
            }
        )
        fig.update_traces(marker=dict(size=4))
    else:
        fig = px.scatter(
            df, x=x, y=y,
            color="front_level",
            animation_frame="generation",
            width=width, height=height,
            title="Pareto Front Evolution (2D)",
            labels={
                x: x.replace('_',' ').title(),
                y: y.replace('_',' ').title(),
                "front_level": "Front Level",
                "generation": "Generation"
            }
        )
        if y_range is not None:
            fig.update_layout(yaxis=dict(range=y_range))
        fig.update_traces(marker=dict(size=6))
    
    fig.update_layout(margin=dict(l=20, r=20, t=50, b=20))
    fig.show()


In [25]:
history = load_pareto_history("experiment_cifar10_nsga/exp1_repeat_1/pareto_history.pkl")
df = pareto_history_to_df("experiment_cifar10_nsga/exp1_repeat_1/pareto_history.pkl", generation=1)
df

,generation,front_level,accuracy,params,inference_time
0,1,1,52.099998,55242.0,3203.630371
1,1,1,68.199997,1201450.0,1221.799805
2,1,1,50.299999,93066.0,2056.980225
3,1,1,70.199997,212042.0,1595.950073
4,1,1,52.400002,121226.0,2088.761230
5,1,1,55.400002,301002.0,686.240173
6,1,2,61.900002,1259914.0,1226.758911
7,1,2,60.799999,275658.0,3340.792725
8,1,2,56.799999,318186.0,2319.955811
9,1,2,64.599998,330762.0,7091.474609


In [31]:
history = load_pareto_history("experiment_cifar10_nsga/exp1_repeat_4/pareto_history.pkl")

In [32]:
# To show 3D animation:
plot_pareto_evolution(history, dims="3d",
                    x="params", y="inference_time", z="accuracy")

In [33]:
# To show 2D animation with Y-axis limited to [0,100]:
plot_pareto_evolution(history, dims="2d",
                    x="params", y="accuracy", y_range=[0,100])